In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output
import pandas as pd

# Sample data with intentional mismatches for checking
wiki_facts = {
    "A. P. J. Abdul Kalam": {
        "infobox": {"Birth Date": "15 May 1980", "Location": "Rameswaram", "Organization": "ISRO"},
        "text": "Born: 16 May 1980 in Rameswaram. He worked at ISRO.",
        "extracted_from_text": {"Birth Date": "16 May 1980", "Location": "Rameswaram", "Organization": "ISRO"}
    },
    "ISRO": {
        "infobox": {"Founder": "Vikram Sarabhai", "Founded": "15 Aug 1969", "Location": "Bengaluru"},
        "text": "ISRO was founded by Vikram Sarabhai on 15 August 1969, headquartered in Bengaluru.",
        "extracted_from_text": {"Founder": "Vikram Sarabhai", "Founded": "15 August 1969", "Location": "Bengaluru"}
    },
    "Chandrayaan-3": {
        "infobox": {"Launch Date": "14 July 2023", "Country": "India"},
        "text": "Launched on 14 July 2023 by India.",
        "extracted_from_text": {"Launch Date": "14 July 2023", "Country": "India"}
    }
}

out = widgets.Output()

def check_fact(article):
    with out:
        clear_output()
        data = wiki_facts[article]
        display(Markdown(f"## 🔍 Fact Check: {article}"))
        display(Markdown(f"**Article Text:** {data['text']}"))

        results = []
        for fact in data['infobox'].keys():
            info_val = data['infobox'][fact]
            text_val = data['extracted_from_text'].get(fact, "Not found")
            match = "✅ Match" if info_val.lower() == text_val.lower() else "⚠️ Possible mismatch detected"
            results.append([fact, info_val, text_val, match])

        df = pd.DataFrame(results, columns=["Fact", "Infobox Value", "Article Text Value", "Result"])
        display(df)

        mismatches = df[df['Result'].str.contains("mismatch")]
        if not mismatches.empty:
            display(Markdown(f"### ⚠️ {len(mismatches)} Possible mismatch for human verification"))
            for _, r in mismatches.iterrows():
                display(Markdown(f"- **{r['Fact']}:** Infobox says `{r['Infobox Value']}` vs Text says `{r['Article Text Value']}`"))
        else:
            display(Markdown("### ✅ All facts consistent"))

drop = widgets.Dropdown(options=list(wiki_facts.keys()), value="A. P. J. Abdul Kalam", description='Select Article:')
widgets.interactive_output(check_fact, {'article': drop})
display(drop, out)
check_fact("A. P. J. Abdul Kalam")

display(Markdown("""
---
**Flow:** Select Article -> Extract Structured Fact -> Search Article Text -> Find Corresponding Info -> Compare Values -> Flag Difference -> Display Both Values
**Note:** This only flags for *human verification*, not final truth.
"""))